### Preliminary cell to start the notebook

In [ ]:
# libraries
import os
import platform
import sys

print(sys.version)

in_colab = "google.colab" in sys.modules
strong_pc = platform.system() == "Linux"

if in_colab:
    if not os.getcwd().split("/")[-1].split("_")[-1] == "2023":
        from google.colab import drive

        drive.mount("/content/drive")
        os.chdir(r"/content/drive/MyDrive/Human_Data_Analytics_Project_2023")

    if not "tensorflow_io" in sys.modules:
        print("Installing tensorflow-IO")
        !pip install tensorflow-io
    if not "keras" in sys.modules:
        print("Installing keras")
        !pip install keras==2.10.0
    if not "scikeras" in sys.modules:
        print("Installing scikeras")
        !pip install scikeras[tensorflow]
    if not "keras-tuner" in sys.modules:
        print("installing keras tuner")
        !pip install keras-tuner
        !pip install numba==0.57.0

main_dir = os.getcwd()
if main_dir not in sys.path:
    print("Adding the folder for the modules")
    sys.path.append(main_dir)

import itertools
import json
import pickle
import random
import shutil
import subprocess
import time
import warnings

import h5py

# PLOT LIBRARIES
import matplotlib
import matplotlib.pyplot as plt

# BASE LIBRARIES
import numpy as np
import pandas as pd

%matplotlib inline
import IPython.display as ipd

# AUDIO LIBRARIES
import librosa
import tensorflow as tf
from keras import layers, models
from keras.utils import plot_model as tf_plot
from scikeras.wrappers import KerasClassifier
from scipy import signal
from scipy.fft import fft, fftfreq, fftshift, ifft
from scipy.io import wavfile
from scipy.signal import periodogram, spectrogram, stft
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier

# MACHINE LEARNING LIBRARIES
from sklearn.model_selection import GridSearchCV, LeaveOneOut, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import check_random_state

# import plotly.express as px


# from pydub import AudioSegment


if in_colab:
    import tensorflow_io as tfio
print("TensorFlow version:", tf.__version__)
# show keras version
import keras

print(f"keras version = {keras.__version__}")
import keras_tuner as kt

# import keras_tune as kt
from keras import layers
from keras.regularizers import L1L2
from tensorflow import keras

# kernel_regularizer=regularizers.L1L2(l1=1e-5, l2=1e-4) # we may use this in some layers...

# RANDOM SETTINGS
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)
check_random_state(seed)

# OUR PERSONAL FUNCTIONS
import importlib

from Models.basic_ml import (
    basic_ML_experiments,
    basic_ML_experiments_gridsearch,
    build_dataset,
    extract_flatten_MFCC,
)
from Preprocessing.data_loader import download_dataset, load_metadata
from Preprocessing.exploration_plots import (
    Spectral_Analysis,
    one_random_audio,
    plot_clip_overview,
)

# EVALUATION LIBRAIRES
from sklearn.metrics import (
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    auc,
    make_scorer,
    precision_recall_curve,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_curve,
)
from Visualization.model_plot import confusion_matrix, listen_to_wrong_audio

importlib.reload(importlib.import_module("Preprocessing.data_loader"))
importlib.reload(importlib.import_module("Models.basic_ml"))
importlib.reload(importlib.import_module("Visualization.model_plot"))

from Models.basic_ml import (
    basic_ML_experiments,
    basic_ML_experiments_gridsearch,
    build_dataset,
    extract_flatten_MFCC,
)
from Preprocessing.data_loader import load_metadata

# df_ESC10, df_ESC50 = load_metadata(main_dir,heads = False, ESC_US = False, statistics=False)


importlib.reload(importlib.import_module("Models.ann_utils"))
importlib.reload(importlib.import_module("Visualization.model_plot"))

from Models.ann_utils import *
from Models.ann_utils import MFCCWithDeltaLayer, OutputCutterLayer
from Visualization.model_plot import (
    confusion_matrix,
    listen_to_wrong_audio,
    plot_history,
    visualize_the_weights,
)

ESC10_path = os.path.join(main_dir, "Data", "ESC-10-depth")
samplerate = 44100

In [ ]:
strong_pc

# 3 UNSUPERVISED LEARNING: AUTOENCODERS

In [ ]:
import importlib

importlib.reload(importlib.import_module("Models.ann_utils"))
importlib.reload(importlib.import_module("Visualization.model_plot"))
importlib.reload(importlib.import_module("Preprocessing.data_loader"))
from Models.ann_utils import *
from Preprocessing.data_loader import reshape_US
from Visualization.model_plot import *

## 3.4 Autoencoder on preprocessed audio STFT on augmented ESC-50 - Convolutional and flatten code

Here we try to train the AE on ESC-50 Augmented instead. We are gonna use ESC-50 to run the grid search of the AE and then we will use ESC-50 Augmented to train the best one. Actually we are goint to use only a small subset of ESC-50 for the gridsearch.

### Create the dataset

In [ ]:
seed = 42
tf.random.set_seed(seed)
ESC50_path = os.path.join(main_dir, "data", "ESC-50-depth")
batch_size = 30
preprocessing = "STFT"
AE_name = "AE_Conv_prep_flatten_" + preprocessing + "_Augmented"

# we are gonna use ESC-50 to run the grid search of the AE and then we will use ESC-50 Augmented to train the best one
train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    ESC50_path,
    verbose=0,
    batch_size=batch_size,
    validation_split=0.25,  # this is the splitting of train vs validation + test
    normalize=True,  # normalization preprocessing (default is true)
    preprocessing=preprocessing,  # "STFT", "MEL", "MFCC" or None
    show_example_batch=True,
    resize=True,
    ndim=3,
)

In [ ]:
# every batch has the shape (batch_size, 128, 64, 1), swap 128 and 64
# in order to have the shape (batch_size, 64, 128, 1)
train = train.map(lambda x, y: (tf.transpose(x, perm=[0, 2, 1, 3]), y))
val = val.map(lambda x, y: (tf.transpose(x, perm=[0, 2, 1, 3]), y))
test = test.map(lambda x, y: (tf.transpose(x, perm=[0, 2, 1, 3]), y))

INPUT_DIM = (64, 128, 1)

In [ ]:
# map the first element of the train dataset over the second and overwrite it
# we do it in order to have the same input and output for the autoencoder
train = train.map(lambda x, y: (x, x))
val = val.map(lambda x, y: (x, x))
test = test.map(lambda x, y: (x, x))

In [ ]:
folder_path = "Saved_Models"  # Replace this with the actual folder path
file_names = [AE_name + "_count.txt"]

for name in file_names:
    file_path = os.path.join(main_dir, folder_path, name)
    with open(file_path, "w") as f:
        f.write("0")
    print(f"Created {name} with content '0' in folder {folder_path}")

### Preparation to use Keras-Tuner

Now we define a function to build a generic convolutional autoencoder. We'll give this function to a keras tuner.

In [ ]:
# General function to build an autoencoder
# CONVOLUTIONAL AUTOENCODER WITH VECTORIAL CODE
code_size = 32
n_layers = 2
n_units = 32


# the real build function for general autoencoder (keras code)
def build_autoencoder(
    img_shape=INPUT_DIM,
    code_size=code_size,
    activation="tanh",
    padding="valid",
    n_layers=n_layers,  # max number of layers is 3
    n_units=n_units,
    kernel_size=(3, 3),
    strides=(2, 2),
    max_pooling=(2, 2),
    regularizer=1e-4,
    batch_norm=True,
    drop_out=0.0,
    learning_rate=1e-3,
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=["mse"],
    AE_name=AE_name,
):
    lr = learning_rate
    # encoder
    encoder = tf.keras.Sequential(name="Encoder")
    encoder.add(tf.keras.Input(img_shape))
    for i in range(n_layers):
        encoder.add(
            layers.Conv2D(
                n_units * (i + 1),
                kernel_size,
                strides=strides,
                activation=activation,
                padding=padding,
            )
        )
        encoder.add(layers.MaxPool2D(max_pooling, padding="same"))
        if batch_norm:
            encoder.add(layers.BatchNormalization())
        if drop_out > 0:
            encoder.add(layers.Dropout(drop_out))

    # flatten layer to get the code
    my_shape = encoder.layers[-1].output_shape
    encoder.add(layers.Flatten())
    encoder.add(
        layers.Dense(
            code_size,
            activation=activation,
            activity_regularizer=keras.regularizers.l1(regularizer),
        )
    )

    # decoder
    decoder = tf.keras.Sequential(name="Decoder")
    decoder.add(tf.keras.Input(code_size))
    decoder.add(layers.Dense(np.prod(my_shape[1:]), activation=activation))
    decoder.add(layers.Reshape(my_shape[1:]))

    # transpose convolutions
    for i in range(n_layers):
        filters = n_units * (n_layers - i) if i < n_layers - 1 else 1
        decoder.add(
            layers.Conv2DTranspose(
                filters,
                kernel_size,
                strides=strides,
                activation=activation,
                padding=padding,
            )
        )
        decoder.add(layers.UpSampling2D(size=max_pooling))
        if batch_norm:
            decoder.add(layers.BatchNormalization())

    # final reshape
    decoder.add(
        tf.keras.layers.Resizing(
            height=INPUT_DIM[0],
            width=INPUT_DIM[1],
            interpolation="bilinear",
            crop_to_aspect_ratio=False,
        )
    )

    # build the autoencoder with keras.Model
    inp = tf.keras.Input(shape=INPUT_DIM)
    code = encoder(inp)
    reconstruction = decoder(code)
    autoencoder = tf.keras.Model(inputs=inp, outputs=reconstruction, name=AE_name)

    # compile the autoencoder
    optimizer = (
        tf.keras.optimizers.Adam(learning_rate=lr)
        if sys.platform == "darwin" or in_colab
        else tf.keras.optimizers.Adam(learning_rate=lr)
    )
    loss = loss
    metrics = metrics

    autoencoder.compile(optimizer=optimizer, loss=loss, metrics=metrics)

    # print the number of trainable parameters
    print(
        f"Model built with {sum(tf.keras.backend.count_params(p) for p in autoencoder.trainable_variables)} trainable params"
    )

    return autoencoder

In [ ]:
verbose = 0
# test the build_autoencoder function
autoencoder = build_autoencoder(n_layers=3)
if verbose > 1:
    autoencoder.summary(line_length=100)
    autoencoder.layers[1].summary(line_length=100)
    autoencoder.layers[2].summary(line_length=100)

In [ ]:
# function to build the model using different hyperparameters (keras tuner code)


def build_model(hp, test=False):
    # define hyperparameters
    if test:  # if test is true you run the tuner only on a reduced hyperparameter space
        print("Running a test smaller grid search")
        n_units = 32
        n_layers = hp.Choice(name="n_layers", values=[2, 3])
        kernel_size = 3
        strides = 2
        max_pooling = 2
        regularizer = hp.Choice(name="regularizer", values=[1e-4, 0.0])
        padding = "same"
        code_size = 32
        activation = "tanh"
        drop_out = hp.Choice(name="drop_out", values=[0.25, 0.0])
        batch_norm = True
        lr_max, lr_min = 1e-3, 1e-3
        hp_lr = hp.Float(
            "learning_rate", min_value=lr_min, max_value=lr_max, sampling="log"
        )
    else:
        n_units = hp.Choice(name="n_units", values=[4, 8, 16, 32, 64, 128], default=32)
        n_layers = hp.Int(
            name="n_layers",
            min_value=1,
            max_value=3,
            step=1,
            sampling="linear",
            default=2,
        )
        kernel_size = hp.Choice(name="kernel_size", values=[3, 5, 7], default=3)
        strides = hp.Choice(name="strides", values=[2, 3], default=2)
        max_pooling = hp.Choice(name="max_pooling", values=[2, 3], default=2)
        regularizer = hp.Choice(
            name="regularizer", values=[0.0, 1e-2, 1e-3, 1e-4, 1e-5], default=1e-4
        )
        padding = hp.Choice(name="padding", values=["same", "valid"], default="valid")
        code_size = hp.Choice(name="code_size", values=[32, 64, 128], default=32)
        activation = hp.Choice(
            name="activation", values=["relu", "elu", "tanh"], default="tanh"
        )
        drop_out = hp.Choice(name="drop_out", values=[0.0, 0.25, 0.5], default=0.0)
        batch_norm = hp.Choice(name="batch_norm", values=[True, False], default=True)
        lr_min, lr_max = 1e-4, 1e-1
        learning_rate = hp.Choice(
            "learning_rate",
            values=[1e-4, 1e-3, 5 * 1e-3, 1e-2, 5 * 1e-2, 1e-1],
            default=1e-3,
        )

    model = build_autoencoder(
        code_size=code_size,
        activation=activation,
        padding=padding,
        n_layers=n_layers,
        n_units=n_units,
        kernel_size=(kernel_size, kernel_size),
        strides=(strides, strides),
        max_pooling=(max_pooling, max_pooling),
        regularizer=regularizer,
        batch_norm=batch_norm,
        drop_out=drop_out,
        learning_rate=learning_rate,
    )

    return model

In [ ]:
# test the build_model function
build_model(kt.HyperParameters()).summary()

### Implement the grid search hyperparamter-wise

In [ ]:
# dictionary with the default values of the hyperparams to be update each time
default_values = {
    "n_layers": 2,
    "n_units": 32,
    "kernel_size": 3,
    "strides": 2,
    "max_pooling": 2,
    "regularizer": 0.0,
    "padding": "valid",
    "code_size": 32,
    "activation": "tanh",
    "drop_out": 0.0,
    "batch_norm": True,
    "learning_rate": 1e-3,
}

key_list = list(default_values.keys())

# define the general variables for our tuner
hpo_methods = ["RandomSearch", "BayesianOptimization", "Hyperband"]
problematic_combination = []
max_model_size = 10**6
max_trials = 10
dir_name = AE_name
verbose = 0

if strong_pc:
    train_small = train
    val_small = val
else:
    small_size_dataset = 400
    train_val_small = train.unbatch().take(small_size_dataset)
    train_small = train_val_small.skip(40).batch(20)
    val_small = train_val_small.take(40).batch(20)

# define a list to collect all the best scores
best_score_dict = {"RandomSearch": [], "BayesianOptimization": [], "Hyperband": []}

# to be consistent with this type of grd search we should pass each hp more than one time...
for hpo_method in hpo_methods:
    random.shuffle(key_list)
    for hyper_params in key_list:
        print(f"Searching for the best value for {hyper_params}")

        # define an hp set with all fix but one
        hp = kt.HyperParameters()

        for fixed_param in default_values.keys():
            if fixed_param != hyper_params:
                hp.Fixed(name=fixed_param, value=default_values[fixed_param])

        if verbose > 1:
            display(hp.space)

        try:
            # create a tuner for the params not fixed
            tuner = build_tuner(
                build_model=build_model,
                hpo_method=hpo_method,
                max_model_size=max_model_size,
                max_trials=max_trials,
                dir_name=dir_name,
                overwrite=True,
                objective=kt.Objective("val_mse", direction="min"),
                hp=hp,
                not_fixed_param=hyper_params,
                tune_new_entries=True,
            )

            if verbose > 2:
                display(tuner.search_space_summary(extended=True))

            # fit the tuner
            epochs = 50
            patience = 10
            metrics = ["mse"]
            callbacks = [
                tf.keras.callbacks.EarlyStopping(
                    monitor="val_" + metrics[0], verbose=verbose, patience=patience
                )
            ]

            tuner.search(
                train_small,
                validation_data=val_small,
                callbacks=callbacks,
                epochs=epochs,
                verbose=int(verbose > 0),
            )

            # retrive the best value for the free hp
            best_value = tuner.get_best_hyperparameters()[0].values[hyper_params]

            # retrive the best score reached
            best_score = tuner.get_best_models(num_models=1)[0].evaluate(
                val, return_dict=True
            )["mse"]

            print(
                f"The best value for {hyper_params} is {best_value}, the best score is {best_score}"
            )
            best_score_dict[hpo_method].append(best_score)

            # update the default dict of values
            default_values[hyper_params] = best_value

            # save the updated dictionary
            file_path = os.path.join(main_dir, dir_name, hpo_method + "_best_params")
            with open(file_path, "wb") as file:
                pickle.dump(default_values, file)

            # delete the folder just created by the run
            shutil.rmtree(
                os.path.join(main_dir, dir_name, hpo_method + "_" + hyper_params)
            )

        except:
            problematic_combination.append(
                ("search_for" + hyper_params, default_values)
            )

    with open(file_path, "rb") as file:
        best_params = pickle.load(file)

    display(best_params)

# save the best_score_dict
file_path = os.path.join(main_dir, dir_name, "best_scores" + preprocessing)
with open(file_path, "wb") as file:
    pickle.dump(best_score_dict, file)

with open(file_path, "rb") as file:
    best_scores = pickle.load(file)

display(best_scores)

display(problematic_combination)

In [ ]:
# compare the best hp from the 3 grid search methods
hyperparamters = []
for hpo_method in ["RandomSearch", "BayesianOptimization", "Hyperband"]:
    file_path = os.path.join(main_dir, dir_name, hpo_method + "_best_params")
    with open(file_path, "rb") as file:
        hyperparamters.append(pickle.load(file))
pd.DataFrame(
    hyperparamters, index=["RandomSearch", "BayesianOptimization", "Hyperband"]
)

### Train the model with the best params on the whole ESC-50 Augmented

In [ ]:
from Models.ann_utils import Masked_AE_training, create_masked_dataset_AE

df_ESC10, df_ESC50 = load_metadata(
    main_dir, heads=False, ESC_US=False, statistics=False
)

In [ ]:
best_params = {
    "n_layers": 3,
    "n_units": 32,
    "kernel_size": 5,
    "strides": 2,
    "max_pooling": 2,
    "regularizer": 0.001,
    "padding": "same",
    "code_size": 32,
    "activation": "elu",
    "drop_out": 0.25,
    "batch_norm": False,
    "learning_rate": 0.001,
}


# build an autoencoder with the best params
autoencoder = build_autoencoder(**best_params)

# autoencoder = tuner.get_best_models(num_models=1)[0] #to create the model with some already wuite good weights
autoencoder.summary()
verbose = 0
if verbose > 0:
    autoencoder.layers[1].summary()
    autoencoder.layers[2].summary()

epochs = 50
n_folders = 20

Masked_AE_training(
    AE_name,
    autoencoder,
    n_folders=n_folders,
    epochs=epochs,
    patience=10,
    verbose=1,
    ndim=3,
    metrics=["mse"],
)

### Show the reconstruction capabilities of the model

In [ ]:
# load the saved model
model_loaded = tf.keras.models.load_model(
    os.path.join(main_dir, "Saved_Models", AE_name)
)
model_loaded.summary()

# plot the original and reconstructed
plot_original_reconstructed(model=model_loaded, n_figures=5, test=test)

Let's try with ESC-10 that has more variability

In [ ]:
train, val, test, label_names = create_dataset(
    ESC10_path,
    verbose=0,
    batch_size=30,
    validation_split=0.25,  # this is the splitting of train vs validation + test
    normalize=True,  # normalization preprocessing (default is true)
    preprocessing=preprocessing,  # "STFT" or "MFCC"
    show_example_batch=False,
    ndim=3,
    resize=True,
    new_width=64,
    new_height=128,
)
model_loaded = tf.keras.models.load_model(
    os.path.join(main_dir, "Saved_Models", AE_name)
)
model_loaded.summary()
# show n original and reconstructed images
plot_original_reconstructed(model=model_loaded, n_figures=5, test=test)